# LAB 03 — Delta DML & Time Travel

> *"Merge new customer data, handle accidental deletes, recover using Time Travel, and see how VACUUM affects it."*

**Goal:** Upsert with `MERGE INTO`, change rows with `UPDATE`, inspect `DESCRIBE HISTORY`, recover from an accidental `DELETE` with time travel and `RESTORE`, and see how `VACUUM` limits time travel.

**Prerequisites:** Notebook attached to **Serverless** compute. `00_setup` only — this lab does not require LAB 02 tables (the Setup cells rebuild `bronze.customers` from `customers.csv`).

| Task | What you do |
|------|-------------|
| Task 1 | Load the update file `customers_new.csv` into `df_updates` |
| Task 2 | `MERGE INTO` the customers table from `v_updates` (update matched, insert new) |
| Task 3 | `UPDATE` the state of the Austin customers to `'TX'` |
| Task 4 | Run the provided accidental `DELETE` (no code to write) |
| Task 5 | Show the table history with `DESCRIBE HISTORY` |
| Task 6 | Query the version before the DELETE with `VERSION AS OF` |
| Task 7 | `RESTORE TABLE` to the version before the DELETE |
| Task 8 | `VACUUM ... RETAIN 0 HOURS`, then confirm that time travel to version 0 fails |

## Setup

In [0]:
%run ../setup/00_setup

In [0]:
# Ensure bronze.customers table exists (idempotent)
customers_path = f"{DATASET_PATH}/customers/customers.csv"
df_base = spark.read.format("csv").option("header", True).option("inferSchema", True).load(customers_path)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")
df_base.write.mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
print(f"Base table ready: {spark.table(f'{CATALOG}.{BRONZE_SCHEMA}.customers').count()} rows")

## Task 1 — Examine the Update File

Load the customer update file and compare its row count with the base table.

**What you need to do:** Use `spark.read` to load `customers_new.csv` into a variable called `df_updates`. The file path is defined for you as `update_path`.

**Expected:** 14 update rows (the base table has 10,000).

In [0]:
update_path = f"{DATASET_PATH}/customers/customers_new.csv"
df_updates = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(update_path)
)

print(f"Existing customers: {spark.table(f'{CATALOG}.{BRONZE_SCHEMA}.customers').count()}")
print(f"Updates file: {df_updates.count()} rows")
display(df_updates.limit(5))

#### Hint — Task 1

CSV format, `header=True`, `inferSchema=True`, `.load(update_path)` — same pattern as the Setup cell that builds the base table.

In [0]:
# -- Validation --
assert df_updates.count() > 0, "Updates file is empty"
print(f"Task 1 OK: {df_updates.count()} update records loaded")

## Task 2 — MERGE INTO (Upsert)

`v_updates` is registered for you in the next cell; write the MERGE. It must:
- **UPDATE** existing customers (match on `customer_id`)
- **INSERT** new customers

**What you need to do:** Write a `MERGE INTO` with the fully qualified target `{CATALOG}.{BRONZE_SCHEMA}.customers` and the source view `v_updates`.

**Expected:** 10,006 rows after the MERGE (8 rows of the update file match existing customers, 6 are new).

In [0]:
# Register updates as temp view
df_updates.createOrReplaceTempView("v_updates")

In [0]:
spark.sql(f"""
    MERGE INTO {CATALOG}.{BRONZE_SCHEMA}.customers AS t
    USING v_updates AS s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

new_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Customers after MERGE: {new_count}")

#### Hint — Task 2

`MERGE INTO` (upsert) applies updates and inserts in a single atomic operation — the standard way to sync Delta tables.

**MERGE syntax**
```sql
MERGE INTO <target> AS t
USING <source> AS s
ON t.key = s.key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

`UPDATE SET *` updates all target columns from the source. `INSERT *` inserts the full source row.

In [0]:
# -- Validation --
base_count = df_base.count()
assert new_count >= base_count, f"Expected at least {base_count} rows after MERGE, got {new_count} — check WHEN MATCHED THEN UPDATE SET * / WHEN NOT MATCHED THEN INSERT *"
print(f"Task 2 OK: MERGE completed. {new_count} total customers (was {base_count})")

## Task 3 — UPDATE Records

Update the `state` column for all customers where `city = 'Austin'`. Set state to `'TX'`.

> **Note:** The base file has no Austin customers — the **Austin rows arrive with the update file** you merged in Task 2, and they carry wrong states (`CA`, `MI`). Your UPDATE fixes them. Expected: **5** Austin rows after the MERGE (4 in the file; one of those IDs appears twice in the base table, so MERGE updates both rows).

**What you need to do:** Write an `UPDATE` on `{CATALOG}.{BRONZE_SCHEMA}.customers` that sets `state = 'TX'` for all rows where `city = 'Austin'`.

In [0]:
spark.sql(f"""
    UPDATE {CATALOG}.{BRONZE_SCHEMA}.customers
    SET state = 'TX'
    WHERE city = 'Austin'
""")

display(spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.customers WHERE city = 'Austin'"))

#### Hint — Task 3

Delta Lake supports `UPDATE` directly — unlike plain Parquet files, you can modify rows in place.

**UPDATE syntax**
```sql
UPDATE catalog.schema.table
SET column = 'value'
WHERE condition = 'value'
```

String literals in SQL must be quoted: `'TX'`, `'Austin'`.

In [0]:
# -- Validation --
austin = spark.sql(f"SELECT state FROM {CATALOG}.{BRONZE_SCHEMA}.customers WHERE city = 'Austin'").collect()
assert len(austin) > 0, "No Austin customers found — did the MERGE in Task 2 run?"
wrong = [r["state"] for r in austin if r["state"] != "TX"]
assert not wrong, f"Some Austin customers still have a state other than 'TX': {wrong}"
print(f"Task 3 OK: {len(austin)} Austin customers, all with state = 'TX'")


## Task 4 — Accidental DELETE (run only, no code to write)

Run the provided cell. It simulates an accident: `DELETE ... WHERE country IS NOT NULL`.

This deletes **every** row (no customer has a NULL country) — you will recover them with time travel.

In [0]:
# Record row count BEFORE the accident
count_before = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Rows BEFORE delete: {count_before}")

# "Accident" - deletes every row (no customer has a NULL country)
spark.sql(f"""
    DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    WHERE country IS NOT NULL
""")

count_after = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Rows AFTER delete: {count_after} (lost {count_before - count_after} rows!)")

## Task 5 — DESCRIBE HISTORY

Check the table history to see all operations performed.

**What you need to do:** Run `DESCRIBE HISTORY` on `{CATALOG}.{BRONZE_SCHEMA}.customers` and wrap it in `display()` so you can read the version numbers. Note the version of the DELETE — you need it in Tasks 6 and 7.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers"))

#### Hint — Task 5

Every write operation on a Delta table creates a new entry in the transaction log with a version number.

**DESCRIBE HISTORY syntax**
```sql
DESCRIBE HISTORY catalog.schema.table
```

Returns one row per version with: `version` (integer), `timestamp`, `operation` (e.g. WRITE / CREATE TABLE AS SELECT, MERGE, UPDATE, OPTIMIZE, DELETE, RESTORE), and operation parameters.

On serverless you may also see **OPTIMIZE** commits you did not run — automatic compaction after a write. They rewrite files but do not change the data.

In [0]:
# -- Validation --
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
operations = [row["operation"] for row in history]
assert "DELETE" in operations, "Expected DELETE in history — did you run the Task 4 cell?"
assert "MERGE" in operations, "Expected MERGE in history — did your MERGE in Task 2 run on this table?"
print(f"Task 5 OK: {len(history)} versions found. Operations: {operations}")

## Task 6 — Time Travel: Query the Previous Version

Read the table as it was BEFORE the accidental delete.

**What you need to do:**
1. In the Task 5 history, find the version of the DELETE.
2. Set `version_before_delete` = the version of the DELETE minus 1 — the commit right before the DELETE: the UPDATE, or on serverless often an automatic **OPTIMIZE** commit (same data as after the UPDATE).
3. Read that version into `df_recovered` using `spark.sql()` with `VERSION AS OF`.

**Expected:** `df_recovered` has 10,006 rows (the current table has 0).

In [0]:
# Find version just before the DELETE automatically
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
delete_version = next(r["version"] for r in history if r["operation"] == "DELETE")
version_before_delete = delete_version - 1

df_recovered = spark.sql(f"""
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    VERSION AS OF {version_before_delete}
""")

print(f"version_before_delete: {version_before_delete}")
print(f"Recovered version has {df_recovered.count()} rows (current has {count_after})")

#### Hint — Task 6

Delta Lake time travel lets you **read** any previous version without modifying the table.

**VERSION AS OF syntax**
```sql
SELECT * FROM catalog.schema.table VERSION AS OF <version_number>
```

Version numbers are integers starting at 0 (the initial table write). You can type the number you read in the history, or look it up in Python from the `DESCRIBE HISTORY` rows (`operation == "DELETE"`).

In [0]:
# -- Validation --
assert df_recovered.count() > count_after, "Recovered version should have more rows than current — did you use version_before_delete (the DELETE version minus 1) in VERSION AS OF?"
print(f"Task 6 OK: Time Travel successful! Recovered {df_recovered.count()} rows")

## Task 7 — RESTORE the Table

Use `RESTORE TABLE` to bring the table back to the version before the accidental delete.

**What you need to do:** Run `RESTORE TABLE ... TO VERSION AS OF` on `{CATALOG}.{BRONZE_SCHEMA}.customers` with the same `version_before_delete` from Task 6.

**Expected:** 10,006 rows — the same count as before the DELETE.

In [0]:
spark.sql(f"""
    RESTORE TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
    TO VERSION AS OF {version_before_delete}
""")

restored_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Rows after RESTORE: {restored_count}")

#### Hint — Task 7

`RESTORE TABLE` rolls the table back to a previous version — it adds a new `RESTORE` entry to the transaction log and makes that version the current state.

**RESTORE syntax**
```sql
RESTORE TABLE catalog.schema.table TO VERSION AS OF <version>
```

After RESTORE, `DESCRIBE HISTORY` shows a new RESTORE operation at the top.

In [0]:
# -- Validation --
assert restored_count == count_before, f"Expected {count_before} rows after restore, got {restored_count} — did you RESTORE ... TO VERSION AS OF version_before_delete?"
print(f"Task 7 OK: Table restored! {restored_count} rows (matches pre-delete count)")

## Task 8 — VACUUM and Its Impact on Time Travel

Run `VACUUM` with 0 hours retention, then try to query an old version. Time travel to the vacuumed versions is **rejected**: Delta refuses to read versions older than `delta.deletedFileRetentionDuration`, because VACUUM is allowed to delete their data files. (On Unity Catalog managed tables the physical deletion may happen later — the retention guard is what you observe.)

> **Note:** `RETAIN 0 HOURS` is for this lab only. The default retention is **7 days** — never lower it in production without understanding the consequences.

Run the provided cell — it prepares the table so VACUUM can remove old files.

In [0]:
# Step 1: Check versions available before VACUUM
history_before = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
print(f"Versions available: {len(history_before)}")
print(f"Version numbers: {[r['version'] for r in history_before]}")

# Step 2: OPTIMIZE — rewrite the data files so the files of old versions are no longer
# referenced by the current version (needed when deletion vectors are enabled)
display(spark.sql(f"OPTIMIZE {CATALOG}.{BRONZE_SCHEMA}.customers"))

# Step 3: Bypass the retention safety check (LAB ONLY — never do this in production!)
try:
    # Classic compute
    spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
    print("Retention safety check disabled via Spark conf (classic compute)")
except Exception as e:
    # Serverless: conf not available (CONFIG_NOT_AVAILABLE) -> lower the table's own retention instead
    spark.sql(f"ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
    print(f"Spark conf not settable here ({type(e).__name__}) — set delta.deletedFileRetentionDuration = 0 hours on the table")


In [0]:
spark.sql(f"VACUUM {CATALOG}.{BRONZE_SCHEMA}.customers RETAIN 0 HOURS")

print("VACUUM complete — unreferenced old data files deleted (UC managed tables may defer the physical cleanup)")

In [0]:
time_travel_failed = None
try:
    df_old = spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.customers VERSION AS OF 0")
    df_old.count()   # action — forces Spark to actually read the files
    time_travel_failed = False
    print("Unexpected: VERSION AS OF 0 is still readable after VACUUM (time travel was not blocked)")
except Exception as e:
    time_travel_failed = True
    print(f"Expected error after VACUUM — time travel to version 0 is rejected by the retention guard (delta.deletedFileRetentionDuration):")
    print(f"  {type(e).__name__}: {str(e)[:200]}")


#### Hint — Task 8

`VACUUM` permanently deletes data files that the current version no longer references and that are older than the retention threshold. Versions that need those files can no longer be queried with time travel.

**VACUUM syntax**
```sql
VACUUM catalog.schema.table RETAIN <hours> HOURS
```

**Time-travel check**
- Query `VERSION AS OF 0` inside `try`. Spark is lazy, so run an **action** (e.g. `.count()`) inside the `try` as well.
- After VACUUM the query raises an error — here `DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION`: the version is older than `delta.deletedFileRetentionDuration`, so Delta blocks it (VACUUM may have deleted its files). Where the table property was not lowered, a missing-file error is possible instead.
- Set `time_travel_failed = True` in the `except` branch and `False` if the query succeeds — the validation checks it.

In [0]:
# -- Validation --
assert time_travel_failed is not None, "Set time_travel_failed in the previous cell"
assert time_travel_failed, ("Time travel to version 0 should be rejected (retention guard) after VACUUM — "
                            "did you run the provided Task 8 cell (OPTIMIZE + retention bypass) and VACUUM ... RETAIN 0 HOURS?")
# After VACUUM, the latest version should still be accessible
current = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
assert current > 0, "Current version should still work after VACUUM"
print(f"Task 8 OK: Current table has {current} rows (latest version OK), time travel to version 0 is rejected")
print("Key takeaway: VACUUM deletes old unreferenced files → Time Travel to vacuumed versions is rejected (retention guard)")
print(f"  Default retention: 7 days | Production best practice: never set to 0 hours")


## Summary

You have:
- Used MERGE INTO for upsert (insert + update)
- Performed UPDATE and DELETE on Delta tables
- Inspected history with DESCRIBE HISTORY
- Queried previous versions with Time Travel
- Restored a table with RESTORE TABLE
- Ran VACUUM and observed its impact on Time Travel

Key takeaways:
- VACUUM deletes old unreferenced data files → time travel to vacuumed versions is rejected
- Default retention is 7 days — never set it to 0 hours in production

> **Exam tip:** Time Travel uses the Delta transaction log plus the data files each version references. Data files for old versions are only removed by `VACUUM`. Default retention is **7 days** (`delta.deletedFileRetentionDuration`); log entries are kept per `delta.logRetentionDuration` (default 30 days). After VACUUM, `DESCRIBE HISTORY` still shows metadata, but querying vacuumed versions fails — Delta rejects versions older than the file retention because their files may be gone (on Unity Catalog managed tables the physical deletion can be deferred).

> **Next:** LAB 04 — Delta Optimization

## Cleanup (Optional)

In [0]:
# Optional cleanup
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")
print("LAB 03 complete.")

← [03 — Delta Lake](../day1/demo/03_delta_lake.ipynb) | **[ README](../../README.md)** | [04 — Delta Optimization →](../day2/demo/04_delta_optimization.ipynb)